# Datasets and Dataloaders

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
X, y = make_classification(n_samples=10, n_features=2, n_informative=2, n_redundant=0, n_classes=2, random_state=42)

In [5]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [6]:
from torch.utils.data import Dataset, DataLoader

In [13]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
       
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [14]:
dataset = CustomDataset(X,y)

In [17]:
print(len(dataset))
print(dataset[2])

10
(tensor([-2.8954,  1.9769]), tensor(0))


In [18]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [21]:
for batch_features , batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print('_'*50)

tensor([[-0.5872, -1.9717],
        [ 1.7273, -1.1858]])
tensor([0, 1])
__________________________________________________
tensor([[ 1.0683, -0.9701],
        [ 1.8997,  0.8344]])
tensor([1, 1])
__________________________________________________
tensor([[ 1.7774,  1.5116],
        [-0.9382, -0.5430]])
tensor([1, 1])
__________________________________________________
tensor([[-2.8954,  1.9769],
        [-1.9629, -0.9923]])
tensor([0, 0])
__________________________________________________
tensor([[-0.7206, -0.9606],
        [-1.1402, -0.8388]])
tensor([0, 0])
__________________________________________________


## Solving Dataset Problem

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [24]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [25]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:,0], test_size=0.2)

In [27]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [28]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [29]:
X_train_tensor = torch.from_numpy(X_train).float()
X_test_tensor  = torch.from_numpy(X_test).float()

y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor  = torch.from_numpy(y_test).float()


In [37]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, features, label):
        self.features = features
        self.label = label
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[0], self.label[0]

In [38]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [40]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset,  batch_size=32, shuffle=True)

In [41]:
class MySimpleNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, X):
        out = self.linear(X)
        out = self.sigmoid(out)
        return out

In [42]:
learning_rate= 0.1
epochs = 25

In [43]:
loss_function = nn.BCELoss()

In [45]:
#create model
model = MySimpleNN(X_train_tensor.shape[1])

optim = torch.optim.SGD(model.parameters(), lr=learning_rate)
#defineloop
for epoch in range(epochs):
   for batch_features, batch_labels in train_loader:
      y_pred = model(batch_features)
      loss = loss_function(y_pred, batch_labels.view(-1,1))
      optim.zero_grad()
      loss.backward
      optim.step()
   
   print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.7489064335823059
Epoch: 2, Loss:0.7489064335823059
Epoch: 3, Loss:0.7489064335823059
Epoch: 4, Loss:0.7489064335823059
Epoch: 5, Loss:0.7489064335823059
Epoch: 6, Loss:0.7489064335823059
Epoch: 7, Loss:0.7489064335823059
Epoch: 8, Loss:0.7489064335823059
Epoch: 9, Loss:0.7489064335823059
Epoch: 10, Loss:0.7489064335823059
Epoch: 11, Loss:0.7489064335823059
Epoch: 12, Loss:0.7489064335823059
Epoch: 13, Loss:0.7489064335823059
Epoch: 14, Loss:0.7489064335823059
Epoch: 15, Loss:0.7489064335823059
Epoch: 16, Loss:0.7489064335823059
Epoch: 17, Loss:0.7489064335823059
Epoch: 18, Loss:0.7489064335823059
Epoch: 19, Loss:0.7489064335823059
Epoch: 20, Loss:0.7489064335823059
Epoch: 21, Loss:0.7489064335823059
Epoch: 22, Loss:0.7489064335823059
Epoch: 23, Loss:0.7489064335823059
Epoch: 24, Loss:0.7489064335823059
Epoch: 25, Loss:0.7489064335823059


In [46]:
model.eval()
accuaracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)
        y_pred = (y_pred>0.8).float()

        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuaracy_list.append(batch_accuracy)
    
overall_accuracy = sum(accuaracy_list)/ len(accuaracy_list)
print(f'Accuracy: {overall_accuracy: .4f}')

Accuracy:  1.0000
